# Data manipulation with Pandas 

First things first: import `pandas`. By the way we import also `numpy`, which is useful for numerical operations.

In [ ]:
import pandas as pd
import numpy as np

## Importing data form files

We typically use Pandas to manipulate data obtained from external sources. 

For instance CSV (comm-separated-values) like this file:

In [ ]:
hospitalsFile = "../../data/health-valais/hospitals.csv"

We can directly read it and import it into a `DataFrame` with this function:

In [ ]:
hospitals=pd.read_csv(hospitalsFile,encoding='latin')
hospitals

### 1. Encoding

Be sure to use the right encoding. We used `latin` in the previous example and many characters were not correctly interpreted. **Now try to change the code to read it using the `utf-8` encoding:**

In [ ]:
hospitals=pd.read_csv(hospitalsFile,encoding='utf-8')

hospitals

## DataFrame metadata and manipulation

The actually containts information about hospitals in Valais. In this case the finle is small but it could be super long. 

We can select for instance the first 3 rows:

In [ ]:
hospitals.head(3)

We can also list the columns:

In [ ]:
hospitals.columns

And the indexes:

In [ ]:
hospitals.index

We can get the number of elements (rows):

In [ ]:
len(hospitals)

Or both rows and columns of the `DataFrame`:

In [ ]:
hospitals.shape

We can also have a summary of the datatypes:

In [ ]:
hospitals.dtypes

To get only one column we can access it by name:

In [ ]:
hospitals['Adresse']

And we can perform some basic statistic operations linke `min()` or `max()`:

In [ ]:
hospitals['beds'].mean()

In [ ]:
hospitals['beds'].max()

A special function provides all basic statistics of the `DataFrame` (in the columns where this is possible)

In [ ]:
hospitals.describe()

Now back to the hospitals. We can create a new column, with only 0s in it:

In [ ]:
hospitals['newBeds']=0
hospitals

And now we can put some contents to that column. Silly example, we'll just put the number of beds multiplied by 2:

In [ ]:
hospitals['newBeds']=hospitals['beds']*2
hospitals.head()

### 2. Remove duplicates

Pandas has a function to identify duplicates based on a selection of columns. Let's find duplicates by Address, number and number of beds:

In [ ]:
hospitals.duplicated(["Adresse","numero","beds"]) 

For each row it indicates `True` if there is a duplicate. **In which row was there a duplicate?**

We can use a Pandas function to drop duplicates automatically. It drops all duplicates according to a subset of columns:

In [ ]:
hospitals.drop_duplicates(["Adresse","numero","beds"]) 

We can see now that the duplicated row has been eliminated. 

### 3. Finding missing values

We can verify with Pandas if there are any missing values in the table (Dataframe). The following code will tell us if there are missing values or not:

In [ ]:
hospitals.isnull().values.any()  

Indeed there are. If you check all the `NaN` values, for example in most of the country values, are missing. 

We can have a more detailed summary of how many missing values are per column:

In [ ]:
hospitals.isnull().sum() 

There are many null values indeed. We can fix this by elimination. 


## Droping an entire column

For example, the `RuleID` column is totally empty. We could get rid of it entirely with the `drop` function:

In [ ]:
hospitals.drop(columns='RuleID', inplace=True)
hospitals.isnull().sum()

## Droping rows with missing values

We can see that there is one row that has no Address. This seems to be a msitake, and we may want to drop the entire row. 

The `dropna` function will help us, it will drop rows that have missing values in the specified column: 

In [ ]:
hospitals.dropna(subset = ['Adresse'], inplace=True)
hospitals.isnull().sum()

Now we can see that all entries have an address. 

## Filling missing values with a default value

We can see that most countries are empty. We can use the `fillna` function to set a fixed value for all rows where it was missing:

In [ ]:
hospitals["country"].fillna('Suisse', inplace=True)
hospitals

## Fill missing values with a computed value

We see that there is one hospital wihtout number of beds. To quickly fix it we will input the minimum of hospital beds into it:

In [ ]:
hospitals['beds'].fillna((hospitals['beds'].min()), inplace=True)
hospitals

## Filtering

There are lots of operators and filtering options in Pandas. We'll see just a handful.

For instance, selecting a range of rows:

In [ ]:
hospitals[5:8]

Or locating a specific row through its index:

In [ ]:
hospitals.loc[11]

We can also combine row selection and column selection:

In [ ]:
hospitals.loc[9:12, ['ETABLISSEMENT','beds']]

We can do all sorts of filtering. For example only take those hospitals in the city of Sion:

In [ ]:
hospitals.loc[(hospitals['ville']=='Sion')] 

Try to find those **hospitals in Sion or in Martigny**

In [ ]:
hopsitals.loc[

We can also find hospitals whose Address starts with a string, for example all that start with the word "Avenue": `hospitals['Adresse'].str.startswith('Avenue')`

Try to **find those hospitals located in an avenue and having less than 15 beds:**


In [ ]:
hospitals.loc[

## Filter out rows when values are out of range

We can drop some rows if we think they have really wrong values. For example the Hospital de Fully has 5000 beds. **Can you update the hospitl list, eliminating all hospotals having more than 2000 beds ?:**

In [ ]:
hospitals=

## Data type modification

We can see that the street number and the number of beds are float values. They should be integer:

In [ ]:
hospitals['numero'] = hospitals['numero'].astype('Int64') 

hospitals

Do the same for the beds:

In [ ]:
hospitals['beds'] = 

## Text modifications

There are othe erros including the `CH-1870` that e can replace. Also the names of some cities are in uppercase (BRIG). We can fix this too:

In [ ]:
hospitals['npa'] = hospitals['npa'].str.replace('CH-','')
hospitals['ville'] = hospitals['ville'].str.title()
hospitals

## Merge column

We can create a new column merging data from others. For example we can create a full address column concatenating the content of the others:

In [ ]:
hospitals['full_adresse'] = hospitals['Adresse'] + ' '+ hospitals['numero'].astype(str) + ', '+ hospitals['npa']+' '+hospitals['ville'] 
hospitals

## Mapping replacement

We can see that some city names in gemran could be modified. We can do a mapping from the old values to new values and apply it to the entire dataframe:

In [ ]:
hospitals=hospitals.replace({"ville":{"Sitten": "Sion"}})
hospitals

**Do it for Visp to Viège**

## Codification

The cities are text entries. We can convert them to categories:

In [ ]:
hospitals['ville'] = pd.Categorical(hospitals.ville)
hospitals.dtypes 

## Grouping

We can also group hospitals by some criteria, for instance by city:

In [ ]:
groups=hospitals.groupby('ville')
groups.get_group('Sion')

Sorting works as expected:

In [ ]:
hospitals.sort_values(by='npa',ascending=False).head()

We can obtain unique values form a column. For instances the cities:

In [ ]:
hospitals['ville'].unique()

And we can even use filtering to create a new data frame and save it to another file:

In [ ]:
hospitalsModif=hospitals.loc[:,['ETABLISSEMENT','newBeds','ville']]
hospitalsModif.to_csv('hospitals_modif.csv',sep=',',encoding='utf-8')

We can also iterate over the dataframe as if it was a list:

In [ ]:
for idx,row in hospitals.iterrows():
    print(row['ville'])
    print(idx)

Grouping is similar to what you would expect in relational databases:

In [ ]:
groups=hospitals.groupby('ville')
list(groups.groups) # or this
groups.get_group('BRIG')